# Day 2.4 — Basic RAG

Retrieval-Augmented Generation means retrieving external evidence and placing it in model context before generation:

```text
Question → retrieve chunks → construct evidence context → model → answer
```

RAG does not train or update the model.

## Before you begin

### Learning outcomes

Trace question to retrieval to grounded generation and diagnose which layer fails.

Architecture reference: [D07](../../diagrams/source/day_02.md).

### Expected observation

The answer uses retrieved evidence; an irrelevant retrieval produces a visibly weak or abstaining result.


## Concept briefing

## Context engineering

Retrieval is one part of context engineering: deciding what the model should see, in
what order, with which labels and within what token budget. A later RAG request may
contain:

```text
system instructions
+ tool descriptions
+ current question
+ selected conversation history
+ retrieved chunks with source labels
+ relevant memory
+ prior tool results
```

Everything included consumes context and can influence generation. Everything excluded
is unavailable to the model. More context is not automatically better; irrelevant or
conflicting material can reduce answer quality. A useful debugging exercise is to print
each component and its approximate token count before sending the request.


In [ ]:
import os,sys
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()
here=Path.cwd().resolve(); candidates=[here,here/"day_02_knowledge_and_state",here.parent]
project_root=next(p for p in candidates if (p/"src"/"knowledge_agent").exists())
sys.path.insert(0,str(project_root/"src"))
from knowledge_agent.documents import load_markdown_corpus
from knowledge_agent.embeddings import SentenceTransformerEmbedder
from knowledge_agent.retrieval import VectorIndex
from openai import OpenAI
chunks=load_markdown_corpus(project_root/"data"/"corpus")
index=VectorIndex(SentenceTransformerEmbedder(os.getenv("EMBEDDING_MODEL","sentence-transformers/all-MiniLM-L6-v2")))
index.add(chunks)
client=OpenAI(base_url="https://openrouter.ai/api/v1",api_key=os.environ["OPENROUTER_API_KEY"])

In [ ]:
question="How long are battery fault records retained?"
retrieved=index.search(question,top_k=3)
context="\n\n".join(
    f"[{x.chunk.chunk_id}] {x.chunk.text}" for x in retrieved
)
print(context)

## Generate only from evidence

Retrieved documents are data, not trusted instructions. The prompt explicitly separates the question and evidence.

In [ ]:
prompt=f"""Answer only from the supplied evidence. If it is insufficient, say so.

Question:
{question}

Evidence:
{context}
"""
response=client.chat.completions.create(
    model=os.getenv("OPENROUTER_MODEL","openai/gpt-oss-120b"),
    messages=[{"role":"user","content":prompt}],
    max_tokens=400,
    extra_body={"reasoning":{"effort":"low","exclude":True}},
)
print(response.choices[0].message.content)

## Break it

Ask for the battery purchase price. Retrieval will still return nearest chunks even though none answers it. A confident instruction is not enough—we need structured citations and explicit abstention.

## Exercise and checkpoint

Print the three chunks used for an answer and identify which actually contains the supporting sentence. RAG has two independently failing stages: retrieval can select poor evidence, and generation can misuse good evidence. Next we enforce citations and abstention.

## Required live observation

Generate one grounded answer with the live model using supplied evidence, then compare it with the deterministic fallback. Do not use live availability as a grading condition.


## Your turn

Replace the top chunk with an irrelevant one and classify the resulting failure.

## Recap

RAG is a pipeline; retrieval and generation must be inspected separately.
